# Generating Embeddings with Hugging Face (Free Models)

**Runs on:** Google Colab (free CPU/GPU), no API key needed.

### What you will learn
1. What an embedding is
2. Generating embeddings with `sentence-transformers`
3. Comparing sentences with cosine similarity
4. Visualising embeddings in 2D
5. Building a mini semantic search engine and an FAQ bot
6. Comparing two embedding models

## 1. What is an Embedding?
An embedding is a list of numbers (a vector) that represents the **meaning** of a text.
Similar meaning gives close vectors, different meaning gives distant vectors.

In [1]:
%pip install -q sentence-transformers scikit-learn matplotlib pandas

## 2. Load a Free Model
`all-MiniLM-L6-v2` is small (~80 MB), fast, and outputs **384-dimensional** vectors.

In [2]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    %pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\tmoun\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2347.53it/s]


Model loaded. Embedding dimension: 384


C:\Users\tmoun\AppData\Local\Temp\ipykernel_7356\3261907094.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())


## 3. Your First Embedding

In [3]:
sentence = "Cricket is the most popular sport in India."
embedding = model.encode(sentence)

print("Sentence:", sentence)
print("Shape:", embedding.shape)
print("First 10 values:", embedding[:10].round(4))

Sentence: Cricket is the most popular sport in India.
Shape: (384,)
First 10 values: [ 0.0688  0.0147 -0.0229 -0.071   0.0407  0.0637  0.0681  0.0513  0.0034
  0.1598]


## 4. Embeddings for Many Sentences

In [4]:
sentences = [
    "The chef is cooking pasta in the kitchen",
    "A cook is preparing noodles",
    "The rocket was launched into space",
    "NASA sent a satellite into orbit",
    "My laptop battery drains very fast",
]

embeddings = model.encode(sentences)
print("Matrix shape (sentences x dimensions):", embeddings.shape)

Matrix shape (sentences x dimensions): (5, 384)


## 5. Similarity Between Sentences

In [6]:
try:
    import pandas as pd
except ModuleNotFoundError:
    %pip install pandas
    import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(embeddings)
pd.DataFrame(sim, index=sentences, columns=range(1, len(sentences) + 1)).round(2)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.1/9.9 MB 1.1 MB/s eta 0:00:09
    --------------------------------------- 0.2/9.9 MB 2.5 MB/s eta 0:00:04
   - -------------------------------------- 0.3/9.9 MB 2.2 MB/s eta 0:00:05
   -- ------------------------------------- 0.6/9.9 MB 3.2 MB/s eta 0:00:03
   --- ------------------------------------ 0.8/9.9 MB 3.5 MB/s eta 0:00:03
   --- ------------------------------------ 0.9/9.9 MB 3.5 MB/s eta 0:00:03
   ---- ----------------------------------- 1.2/9.9 MB 3.7 MB/s eta 0:00:03
   ---- ----------------------------------- 1.2/9.9 MB 3.7 MB/s eta 0:00:03
   ----- ---------------------------------- 1.4/9.9 MB 3.4 MB/s eta 0:00:03
   ------ --------------------------------- 1.5/9.9 MB 3.5 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/9.9 MB 3.3 MB/s eta 0:00:03
   ------ --------------------------------- 1.7/9.9 MB 3.1 MB/s eta 0:00:03
   ------- --------

,1,2,3,4,5
The chef is cooking pasta in the kitchen,1.00,0.65,-0.02,0.03,-0.00
A cook is preparing noodles,0.65,1.00,0.01,0.03,-0.03
The rocket was launched into space,-0.02,0.01,1.00,0.60,-0.01
NASA sent a satellite into orbit,0.03,0.03,0.60,1.00,0.05
My laptop battery drains very fast,-0.00,-0.03,-0.01,0.05,1.00


The cooking pair (1, 2) and the space pair (3, 4) should score noticeably higher with each other than with the laptop sentence.

## 6. Visualise in 2D with PCA

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

points = PCA(n_components=2).fit_transform(embeddings)

plt.figure(figsize=(8, 5))
plt.scatter(points[:, 0], points[:, 1])
for i, s in enumerate(sentences):
    plt.annotate(s, (points[i, 0], points[i, 1]), fontsize=8)
plt.title("Sentence embeddings in 2D (PCA)")
plt.grid(True)
plt.show()

## 7. Mini Semantic Search Engine

In [ ]:
# Our small "document" database
documents = [
    "Python is widely used for artificial intelligence and data science.",
    "Machine learning enables computers to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "SQL is used to store and retrieve information from databases.",
    "Cloud computing provides computing resources through the internet.",
    "Natural language processing helps computers understand human language.",
    "Computer vision allows computers to analyze images and videos.",
    "Generative AI can create text, images, and computer code.",
    "Java is commonly used for backend application development.",
    "Data analytics helps organizations find useful patterns in data."
]

doc_embeddings = model.encode(documents)

print("Embeddings generated successfully!")
print("Number of documents:", len(doc_embeddings))

def semantic_search(query, top_k=2):
    q = model.encode([query])
    scores = cosine_similarity(q, doc_embeddings)[0]
    best = scores.argsort()[::-1][:top_k]
    print(f"Query: {query}")
    for i in best:
        print(f"  {scores[i]:.3f} | {documents[i]}")
    print()

semantic_search("How do deep learning models learn?")
semantic_search("Which cricketer made a hundred?")
semantic_search("How do plants make food?")

NameError: name 'model' is not defined

## 8. Exercise: Simple FAQ Bot
Store question and answer pairs, embed the **questions**, and return the answer of the closest question.

In [9]:
# 5 FAQ questions and answers
faq = {
    "What is machine learning?":
        "Machine learning allows computers to learn patterns from data.",

    "What is Python used for?":
        "Python is widely used for artificial intelligence, data science, and software development.",

    "What is SQL?":
        "SQL is used to store, retrieve, and manage information in databases.",

    "What is cloud computing?":
        "Cloud computing provides computing resources such as servers and storage through the internet.",

    "What is generative AI?":
        "Generative AI can create content such as text, images, and computer code."
}

# Separate questions and answers
faq_questions = list(faq.keys())
faq_answers = list(faq.values())

# Create embeddings for FAQ questions
faq_embeddings = model.encode(faq_questions)


# FAQ bot function
def faq_bot(question):
    question_embedding = model.encode([question])

    scores = cosine_similarity(
        question_embedding,
        faq_embeddings
    )[0]

    best_index = scores.argmax()

    print("Question:", question)
    print("Matched FAQ:", faq_questions[best_index])
    print("Answer:", faq_answers[best_index])
    print("Similarity:", round(scores[best_index], 3))


# Test the FAQ bot
faq_bot("How do computers learn from information?")

faq_bot("What can Python be used for?")

faq_bot("How can I store information in a database?")

Question: How do computers learn from information?
Matched FAQ: What is machine learning?
Answer: Machine learning allows computers to learn patterns from data.
Similarity: 0.508
Question: What can Python be used for?
Matched FAQ: What is Python used for?
Answer: Python is widely used for artificial intelligence, data science, and software development.
Similarity: 0.934
Question: How can I store information in a database?
Matched FAQ: What is SQL?
Answer: SQL is used to store, retrieve, and manage information in databases.
Similarity: 0.429


## 9. Exercise: Compare Two Models

In [8]:
# Load the larger embedding model
model_mpnet = SentenceTransformer("all-mpnet-base-v2")

# Generate embeddings for the same documents
mpnet_embeddings = model_mpnet.encode(documents)

print("MPNet model loaded!")
print("Embedding dimension:", model_mpnet.get_sentence_embedding_dimension())


# Semantic search using MPNet
def semantic_search_mpnet(query, top_k=2):
    query_embedding = model_mpnet.encode([query])

    scores = cosine_similarity(
        query_embedding,
        mpnet_embeddings
    )[0]

    ranked_idx = scores.argsort()[::-1][:top_k]

    print("\nQuery:", query)
    print("-" * 60)

    for idx in ranked_idx:
        print(f"Score: {scores[idx]:.3f} | {documents[idx]}")


# Test the same query with MPNet
semantic_search_mpnet("How do computers learn from data?")

c:\Users\tmoun\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tmoun\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\tmoun\AppData\Local\Programs\Python\Python311\Lib\site-packages\h

MPNet model loaded!
Embedding dimension: 768

Query: How do computers learn from data?
------------------------------------------------------------
Score: 0.602 | Machine learning enables computers to learn patterns from data.
Score: 0.443 | Deep learning uses neural networks with multiple layers.


C:\Users\tmoun\AppData\Local\Temp\ipykernel_7356\2978727542.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model_mpnet.get_sentence_embedding_dimension())


In [7]:
# Install required libraries
!pip install -q sentence-transformers scikit-learn matplotlib

# Import libraries
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load Hugging Face embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Model loaded!")
print("Embedding dimension:", model.get_sentence_embedding_dimension())


# 10 custom documents
documents = [
    "Python is widely used for artificial intelligence and data science.",
    "Machine learning enables computers to learn patterns from data.",
    "Deep learning uses neural networks with multiple layers.",
    "SQL is used to store and retrieve information from databases.",
    "Cloud computing provides computing resources through the internet.",
    "Natural language processing helps computers understand human language.",
    "Computer vision allows computers to analyze images and videos.",
    "Generative AI can create text, images, and computer code.",
    "Java is commonly used for backend application development.",
    "Data analytics helps organizations find useful patterns in data."
]

# Generate embeddings
doc_embeddings = model.encode(documents)

print("Embeddings generated!")
print("Number of documents:", len(doc_embeddings))


# Semantic search function
def semantic_search(query, top_k=2):
    query_embedding = model.encode([query])

    scores = cosine_similarity(
        query_embedding,
        doc_embeddings
    )[0]

    ranked_idx = scores.argsort()[::-1][:top_k]

    print("\nQuery:", query)
    print("-" * 60)

    for idx in ranked_idx:
        print(f"Score: {scores[idx]:.3f} | {documents[idx]}")


# Test semantic search
semantic_search("How do computers learn from data?")
semantic_search("How can computers understand human language?")
semantic_search("How is information stored in databases?")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2835.66it/s]


Model loaded!
Embedding dimension: 384
Embeddings generated!
Number of documents: 10

Query: How do computers learn from data?
------------------------------------------------------------
Score: 0.651 | Machine learning enables computers to learn patterns from data.
Score: 0.442 | Natural language processing helps computers understand human language.

Query: How can computers understand human language?
------------------------------------------------------------
Score: 0.685 | Natural language processing helps computers understand human language.
Score: 0.432 | Computer vision allows computers to analyze images and videos.

Query: How is information stored in databases?
------------------------------------------------------------
Score: 0.708 | SQL is used to store and retrieve information from databases.
Score: 0.415 | Data analytics helps organizations find useful patterns in data.


C:\Users\tmoun\AppData\Local\Temp\ipykernel_7356\2950224494.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## Key Takeaways

| Concept | Meaning |
|---|---|
| Embedding | Numeric vector representing the meaning of text |
| `all-MiniLM-L6-v2` | Free, small, fast, 384 dimensions |
| `all-mpnet-base-v2` | Free, larger, usually more accurate, 768 dimensions |
| Cosine similarity | Measures closeness of two embeddings |
| Semantic search | Retrieves by meaning, not exact words |
| PCA | Only for visualising high-dimensional vectors in 2D |